## 1. Instalar Apache Airflow en tu venv

Estando **dentro de tu venv** (`(venv)` se ve en el prompt):

1. Elegí una versión de Airflow, por ejemplo 3.1.0 (podés cambiarla si tu consigna especifica otra):

   ```bash
   AIRFLOW_VERSION=3.1.0
   PYTHON_VERSION=3.12
   ```

2. Corré el `pip install` con constraints (como indica la consigna):

   ```bash
   pip install "apache-airflow[celery]==${AIRFLOW_VERSION}" \
     --constraint "https://raw.githubusercontent.com/apache/airflow/constraints-${AIRFLOW_VERSION}/constraints-${PYTHON_VERSION}.txt"
   ```

   Si no te interesa `celery`, podés usar simplemente:

   ```bash
   pip install "apache-airflow==${AIRFLOW_VERSION}" \
     --constraint "https://raw.githubusercontent.com/apache/airflow/constraints-${AIRFLOW_VERSION}/constraints-${PYTHON_VERSION}.txt"
   ```

3. Verificá la instalación:

   ```bash
   airflow version
   ```

   Si esto imprime algo como `3.1.0`, estás OK.

---

## 2. Levantar Airflow en modo standalone

Desde cualquier directorio dentro del venv:

```bash
airflow standalone
```

Esto hace varias cosas automáticamente:

* Inicializa la base de datos.
* Crea un usuario admin.
* Levanta el webserver en `http://localhost:8080`.

En la terminal te va a aparecer un mensaje importante con las credenciales, o algo como:

> Password for the admin user has been previously generated in
> `/home/user/airflow/simple_auth_manager_passwords.json.generated`

Tenés que:

1. Abrir ese archivo que te indica (con `cat`, `less`, `nano`, etc.):

   ```bash
   cat /home/user/airflow/simple_auth_manager_passwords.json.generated
   ```

2. Buscar el usuario (generalmente `admin`) y la contraseña generada.

3. Ir a `http://localhost:8080` desde el navegador y loguearte con esos datos.

---

## 3. Ubicación de los DAGs

Por defecto, Airflow busca DAGs en:

* Linux/macOS: `~/airflow/dags`
* Windows: `C:\Users\<Username>\airflow\dags`

Como vos parecés estar en Linux, lo más probable es:

```bash
cd ~/airflow/dags
```

Si la carpeta no existe, creala:

```bash
mkdir -p ~/airflow/dags
```

---

## 4. Crear el DAG de ejemplo (tutorial modificado)

La documentación del tutorial está acá (lo que te piden recrear):
`https://airflow.apache.org/docs/apache-airflow/stable/tutorial/fundamentals.html`

En vez de llamarlo `tutorial.py` (que probablemente ya existe como ejemplo), usá otro nombre. Algo como:

* Nombre del archivo: `mi_tutorial.py`
* Y cambiá el `dag_id` para que no sea `tutorial` (por ej. `mi_tutorial_dag`).

Ejemplo simple adaptado (estructura muy parecida a la del tutorial oficial):

```python
from __future__ import annotations

from datetime import datetime, timedelta

from airflow import DAG
from airflow.operators.empty import EmptyOperator
from airflow.operators.bash import BashOperator

with DAG(
    dag_id="mi_tutorial_dag",
    description="DAG de práctica basado en el tutorial oficial",
    schedule_interval=timedelta(days=1),
    start_date=datetime(2025, 1, 1),
    catchup=False,
    tags=["mia", "practica", "tutorial"],
) as dag:

    start = EmptyOperator(
        task_id="start",
    )

    print_date = BashOperator(
        task_id="print_date",
        bash_command="date",
    )

    print_hello = BashOperator(
        task_id="print_hello",
        bash_command="echo 'Hola Airflow desde MIA!'",
    )

    end = EmptyOperator(
        task_id="end",
    )

    # Definición de dependencias (orden de ejecución)
    start >> print_date >> print_hello >> end
```

Guardá esto como:

```bash
~/airflow/dags/mi_tutorial.py
```

---

## 5. Verificar que el DAG aparece

1. Con Airflow corriendo (`airflow standalone`):

   * Abrí la UI: `http://localhost:8080`
   * En la pestaña **DAGs** buscá `mi_tutorial_dag` (o el nombre que le pusiste).

2. Desde terminal, también podés ver los DAGs:

   ```bash
   airflow dags list
   ```

   Deberías ver tu `mi_tutorial_dag` en la lista.

---

## 6. Ejecutar el DAG y ver logs

En la UI:

1. Activá el DAG con el toggle (On/Off).
2. Podés:

   * Clickear en el nombre del DAG.
   * Ir a **Graph** o **Grid**.
   * Clickear en cada tarea (`print_date`, `print_hello`, etc.).
   * En el pop-up, ir a **Log** para ver qué se ejecutó.

También desde terminal, por ejemplo:

```bash
airflow tasks test mi_tutorial_dag print_hello 2025-01-01
```


```
/home/matias/airflow/dags
```